In [0]:
import pkg_resources

In [0]:
%pip list

In [0]:
!pip install -U --quiet PyMUPDF langchain databricks-langchain

In [0]:
%pip install --upgrade typing_extensions

In [0]:
%restart_python

In [0]:
file_path = "/Volumes/workspace/udemy/demovol/dpact.pdf"

import fitz  # PyMuPDF

# Read file content
with open(file_path, "rb") as f:
    pdf_bytes = f.read()

# Use PyMuPDF to extract text
doc = fitz.open("pdf", pdf_bytes)

text = ""
for page in doc:
    text += page.get_text()

print(text)  # Print first 1000 characters

# CSV Example
df = spark.read.option("header", "true").csv("/Volumes/<catalog>/<schema>/<table>")

# OR Parquet
# df = spark.read.parquet("/Volumes/<catalog>/<schema>/<table>")

# OR JSON
# df = spark.read.option("multiline", "true").json("/Volumes/<catalog>/<schema>/<table>")

In [0]:
%pip list | grep databricks-langchain

In [0]:
%pip list

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import pandas as pd

# Load documents (e.g., text files or PDFs from DBFS or local)
raw_text = text
# Replace this with your actual file reader logic

# Chunk documents for embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.create_documents([raw_text])

# Convert docs to a list of dicts for display
pd_docs = pd.DataFrame([doc.dict() for doc in docs])

# Add an index column
pd_docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))

display(pd_docs)

In [0]:
len(docs)

In [0]:
docs[1]

In [0]:
# Create Spark DataFrame from pandas DataFrame
spark_df = spark.createDataFrame(pd_docs[['id_pk', 'page_content']])

# Write to Delta table
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.udemy.my_data_chunks")

In [0]:
# Import and initialize Vector Search Client
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# [NOTICE] Using a notebook authentication token.
# Recommended for development only.
# For improved performance, use Service-based authentication.
# To disable this message, pass disable_notice=True.

In [0]:
client.create_endpoint(
    name="vs_endpoint1",
    endpoint_type="STANDARD"  # or "STORAGE_OPTIMIZED"
)

In [0]:
%sql
ALTER TABLE workspace.udemy.my_data_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%sql
ALTER TABLE workspace.udemy.my_data_chunks
SET TBLPROPERTIES ('delta.change.data.capture' = 'true')

In [0]:
spark.databricks.delta.allowArbitraryProperties.enabled true


In [0]:
# creates Vector Search index named documents_index, and sets - Vector Search endpoint

vs_index = client.create_delta_sync_index(
    endpoint_name="vs_endpoint1",
    index_name="workspace.udemy.udemy_doc_index",
    source_table_name="workspace.udemy.my_data_chunks",  
    primary_key="id_pk",
    embedding_source_column="page_content",
    embedding_model_endpoint_name="udemy_demo_rag_endpoint",
    pipeline_type="TRIGGERED"
)